# Phase 2: Exploratory Data Analysis & Daily Sales Aggregation
**Project:** Retail Sales & Demand Forecasting (FUTURE_ML_01)  
**Objective:** Ingest, inspect, clean, and analyze transactional retail sales data to prepare a robust daily demand time-series for forecasting models.

## 1. Environment Setup & Component Imports
We import modular components from `src.data_loader`, `src.cleaner`, and `src.eda`.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.data_loader import load_raw_data, validate_raw_data, format_validation_summary, RAW_DATA_DIR, DEFAULT_EXCEL_FILENAME
from src.cleaner import clean_transactions, aggregate_daily_sales, export_processed_data

# Display settings
pd.set_option('display.max_columns', 20)
pd.set_option('display.width', 1000)
%matplotlib inline

## 2. Raw Dataset Ingestion & Validation
We load the downloaded spreadsheet without dropping records and run non-destructive schema validation.

In [ ]:
raw_path = RAW_DATA_DIR / DEFAULT_EXCEL_FILENAME
raw_df = load_raw_data(raw_path)
report = validate_raw_data(raw_df)
print(format_validation_summary(report))

### Key Findings from Raw Validation:
- **Dimensions:** 541,909 rows, 8 columns.
- **Date Range:** 2010-12-01 08:26:00 to 2011-12-09 12:50:00 (~374 calendar days).
- **Missing Values:** `CustomerID` has 135,080 missing entries (24.93%) representing guest checkouts. `Description` has 1,454 missing values (0.27%), which overlap 100% with administrative adjustment records.
- **Duplicates:** 5,268 exact duplicate rows (0.97%).
- **Anomalies:** 10,624 negative quantities (cancellations/returns) and 2,517 unit prices <= 0 (warehouse write-offs & bad debt entries).

## 3. Data Cleaning Pipeline
We apply our transparent, documented business rules:
1. **Deduplication:** Remove exact duplicate transaction entries.
2. **Warehouse Write-offs Filter:** Exclude records where `UnitPrice <= 0` (damaged stock, test audits, and accounting debt adjustments).
3. **Guest Checkout Preservation:** Retain missing `CustomerID` by imputing `-1`. Dropping them would discard 25% of genuine retail sales demand.
4. **Cancellation Auditing:** Keep cancellations (`InvoiceNo` starting with 'C') and calculate `LineTotal = Quantity * UnitPrice` to enable accurate net sales calculation.

In [ ]:
cleaned_df = clean_transactions(raw_df)
print(f"Initial records: {len(raw_df):,}")
print(f"Cleaned records: {len(cleaned_df):,} ({len(cleaned_df)/len(raw_df)*100:.2f}% retained)")
cleaned_df.head()

## 4. Daily Sales Aggregation
We aggregate individual basket lines into a continuous daily time series.
- **Gross Sales:** Positive purchases.
- **Returns Value:** Total monetary value of returned products.
- **Net Sales:** Gross Sales minus Returns Value.
- **Net Quantity:** Total units bought minus units returned.

In [ ]:
daily_df = aggregate_daily_sales(cleaned_df, fill_missing_dates=True)
export_path = export_processed_data(daily_df)
print(f"Aggregated into {len(daily_df)} daily records from {daily_df.index.min().date()} to {daily_df.index.max().date()}.")
daily_df.head(10)

## 5. Statistical Summary of Daily Sales

In [ ]:
summary_table = pd.DataFrame({
    'Metric': [
        'Total Net Revenue',
        'Total Gross Revenue',
        'Total Returns Value',
        'Total Net Units Sold',
        'Total Calendar Days',
        'Zero-Sales Days (e.g. Saturdays)',
        'Daily Mean Net Revenue',
        'Daily Median Net Revenue',
        'Peak Single-Day Net Revenue'
    ],
    'Value': [
        f"GBP {daily_df['net_sales'].sum():,.2f}",
        f"GBP {daily_df['gross_sales'].sum():,.2f}",
        f"GBP {daily_df['returns_amount'].sum():,.2f}",
        f"{daily_df['net_quantity'].sum():,.0f} units",
        f"{len(daily_df)} days",
        f"{(daily_df['net_sales'] == 0).sum()} days",
        f"GBP {daily_df['net_sales'].mean():,.2f}",
        f"GBP {daily_df['net_sales'].median():,.2f}",
        f"GBP {daily_df['net_sales'].max():,.2f} ({daily_df['net_sales'].idxmax().date()})"
    ]
})
display(summary_table)

## 6. Visual Exploratory Analysis
Below are the core findings visualized from the dataset.

### 1. Missing Data Distribution
![Missing Data Profile](../reports/figures/01_missing_data_profile.png)
- `CustomerID` has 24.93% missing values representing guest checkout transactions.
- Retaining these transactions preserves over GBP 1.4M in retail consumer demand.
- Missing `Description` (0.27%) overlaps 100% with zero-price administrative records.

### 2. Daily Sales Trend & Q4 Peak
![Daily Sales Trend](../reports/figures/02_daily_net_sales_trend.png)
- Daily sales fluctuate around an average of GBP 26,065/day.
- Strong seasonality is visible starting in September 2011, peaking on **November 14, 2011 at GBP 111,958.62**.
- This peak corresponds to wholesale pre-Christmas gift inventory purchasing.

### 3. Day of Week Seasonality & Saturday Closure
![Day of Week Seasonality](../reports/figures/03_day_of_week_seasonality.png)
- The retailer operates Sunday through Friday and is **consistently closed on Saturdays** (yielding 0 sales across all 53 Saturdays in the dataset).
- Thursday is the highest volume weekday (averaging ~GBP 38,500/day).

### 4. Monthly Revenue Dynamics
![Monthly Sales Trend](../reports/figures/04_monthly_sales_trend.png)
- Monthly revenue peaks in November 2011 at GBP 1.46M (670,000 units sold).
- December 2011 only contains 9 days of records before the recording period ended.

### 5. Product Concentration (Pareto Principle)
![Top 10 Products](../reports/figures/05_top_10_products_by_revenue.png)
- Top sellers include 'DOTCOM POSTAGE', 'REGENCY CAKESTAND 3 TIER', and 'WHITE HANGING HEART T-LIGHT HOLDER'.
- Catalog demand follows a classic long-tail Pareto distribution.

### 6. Sales vs. Returns
![Sales vs Cancellations](../reports/figures/06_sales_vs_cancellations.png)
- Total customer returns amount to GBP 893,980 (8.40% of gross revenue).
- Return rates remain stable between 6% and 12% across operating months.

## 7. Key Conclusions & Readiness for Phase 3
1. **Continuous Daily Series:** `data/processed/daily_sales.csv` provides a 374-day cleaned daily time series ready for feature engineering.
2. **Seasonality Indicators:** Weekend/Saturday closure flags and day-of-week one-hot features will be crucial for forecasting models.
3. **Holiday Trends:** Rolling 7-day and 30-day indicators are needed to capture holiday momentum without overfitting.